In [3]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
import time
import mediapipe as mp
from datetime import datetime
import json
import os
from pyzbar import pyzbar
import pyttsx3
import threading
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage

# CONFIGURATION

RESULT_FOLDER = "result/"
CAPTURES_FOLDER = os.path.join(RESULT_FOLDER, "captures/")
DATA_FILE = os.path.join(RESULT_FOLDER, "captures_data.json")

# NEW: CSV and QR folder paths
STUDENTS_CSV = "student_csv/students.csv" 
QR_CODES_FOLDER = "student_qr_codes/"    

MODEL_PATH = os.path.join(RESULT_FOLDER, "palm_back_classifier.keras")

# EMAIL CONFIGURATION 
SENDER_EMAIL = "xinyaoteoh@gmail.com"
SENDER_PASSWORD = "nqezfevpxwhwbbxp"

# Timing
STABLE_TIME = 3.0 
COUNTDOWN_TIME = 3.0
HOVER_TIME = 2.0
CONFIDENCE_THRESHOLD = 0.40

os.makedirs(CAPTURES_FOLDER, exist_ok=True)

# DATABASE FUNCTIONS 
def load_students_from_csv():
    """Load students from CSV file"""
    students = {}
    try:
        import csv
        with open(STUDENTS_CSV, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                student_id = row['student_id'].strip()
                students[student_id] = {
                    'student_id': student_id,
                    'name': row['student_name'].strip(),
                    'email': row['email'].strip(),
                    'programme': row['programme'].strip(),
                    'faculty': row['faculty'].strip()
                }
        print(f"✓ Loaded {len(students)} students from CSV")
        return students
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return {}

def get_student_by_qr(qr_data, students_db):
    """
    Match QR code data with student database
    qr_data should be the student_id (e.g., "24PMR02576")
    """
    student_id = qr_data.strip()
    
    # Check if QR code image exists in folder
    qr_image_path = os.path.join(QR_CODES_FOLDER, f"{student_id}.png")
    if not os.path.exists(qr_image_path):
        # Try with .jpg extension
        qr_image_path = os.path.join(QR_CODES_FOLDER, f"{student_id}.jpg")
        if not os.path.exists(qr_image_path):
            print(f"⚠️ QR code image not found for: {student_id}")
            return None
    
    # Check if student exists in database
    if student_id in students_db:
        print(f"✓ Found student: {students_db[student_id]['name']}")
        return students_db[student_id]
    else:
        print(f"⚠️ Student ID not found in database: {student_id}")
        return None
    try:
        with open(STUDENTS_DB, 'r') as f:
            students = json.load(f)
        return students.get(qr_data)
    except:
        return None


# EMAIL SYSTEM

def send_graduation_email(student_email, student_name, picture_id, image_path):
    """
    Send graduation photo via email with improved error handling
    """
    print(f"\n{'='*70}")
    print(f"📧 Attempting to send email...")
    print(f"   To: {student_email}")
    print(f"   From: {SENDER_EMAIL}")
    print(f"   Picture: {picture_id}")
    print(f"{'='*70}")
    
# Validation - check for placeholder values
    if SENDER_EMAIL == "your_email@gmail.com":
        print("❌ ERROR: Email not configured!")
        print("   Please update SENDER_EMAIL in the code")
        return False
    
    if SENDER_PASSWORD == "your_app_password":
        print("❌ ERROR: App password not configured!")
        print("   Please update SENDER_PASSWORD in the code")
        return False
    
    if not os.path.exists(image_path):
        print(f"❌ ERROR: Image file not found: {image_path}")
        return False
    
    try:
        # Create email message
        msg = MIMEMultipart()
        msg['From'] = SENDER_EMAIL
        msg['To'] = student_email
        msg['Subject'] = f"🎓 Your Graduation Photo - {student_name}"
        
        # HTML email body
        body = f"""
        <html>
        <body style="font-family: Arial, sans-serif; background-color: #f5f5f5; padding: 20px;">
            <div style="max-width: 600px; margin: 0 auto; background: white; border-radius: 10px; overflow: hidden; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
                <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; text-align: center;">
                    <h1 style="color: white; font-size: 42px; margin: 0;">🎓</h1>
                    <h2 style="color: white; margin: 10px 0 0 0;">Happy Graduation!</h2>
                </div>
                
                <div style="padding: 40px;">
                    <h2 style="color: #333;">Dear {student_name},</h2>
                    
                    <p style="font-size: 16px; color: #555; line-height: 1.6;">
                        Congratulations on your graduation! 🎉 We're thrilled to share your official 
                        graduation photo with you. This moment marks the beginning of an exciting new 
                        chapter in your life.
                    </p>
                    
                    <div style="background: #f8f9fa; padding: 20px; margin: 20px 0; border-radius: 8px; border-left: 4px solid #667eea;">
                        <p style="margin: 5px 0; color: #333;">
                            <strong>Picture ID:</strong> {picture_id}
                        </p>
                        <p style="margin: 5px 0; color: #333;">
                            <strong>Date:</strong> {datetime.now().strftime("%B %d, %Y at %I:%M %p")}
                        </p>
                    </div>
                    
                    <p style="font-size: 16px; color: #555; line-height: 1.6;">
                        Your graduation photo is attached to this email. We wish you all the best 
                        in your future endeavors!
                    </p>
                    
                    <div style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #eee;">
                        <p style="color: #999; font-size: 14px; margin: 5px 0;">Best wishes,</p>
                        <p style="color: #667eea; font-size: 16px; font-weight: bold; margin: 5px 0;">
                            University Graduation Team
                        </p>
                    </div>
                </div>
            </div>
        </body>
        </html>
        """
        
        msg.attach(MIMEText(body, 'html'))
        
        # Attach image
        print("   📎 Attaching image...")
        with open(image_path, 'rb') as f:
            img_data = f.read()
        image = MIMEImage(img_data, name=f"graduation_{picture_id}.jpg")
        msg.attach(image)
        print("   ✓ Image attached")
        
        # Connect to Gmail SMTP server
        print("   🔌 Connecting to Gmail...")
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.set_debuglevel(0)  # Set to 1 for detailed debug info
        
        print("   🔐 Starting TLS encryption...")
        server.starttls()
        
        print("   🔑 Logging in...")
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        print("   ✓ Login successful")
        
        print("   📤 Sending email...")
        server.send_message(msg)
        server.quit()
        
        print(f"   ✅ EMAIL SENT SUCCESSFULLY!")
        print(f"{'='*70}\n")
        return True
        
    except smtplib.SMTPAuthenticationError:
        print("\n❌ AUTHENTICATION ERROR!")
        print("   Possible causes:")
        print("   1. Wrong email or password")
        print("   2. Not using App Password (must enable 2-Step Verification)")
        print("   3. App Password has spaces (remove them)")
        print(f"{'='*70}\n")
        return False
        
    except smtplib.SMTPException as e:
        print(f"\n❌ SMTP ERROR: {e}")
        print("   Check your internet connection and Gmail settings")
        print(f"{'='*70}\n")
        return False
        
    except Exception as e:
        print(f"\n❌ UNEXPECTED ERROR: {e}")
        print(f"   Error type: {type(e).__name__}")
        print(f"{'='*70}\n")
        return False        


# LOAD MODEL
print("🔄 Loading model...")
model = keras.models.load_model(MODEL_PATH)
print("✓ Model loaded")

# MEDIAPIPE
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

# HAND DETECTION
def detect_hand(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    
    if not results.multi_hand_landmarks:
        return None, None, None
    
    landmarks = results.multi_hand_landmarks[0]
    h, w = frame.shape[:2]
    
    x_min, y_min = w, h
    x_max, y_max = 0, 0
    
    for lm in landmarks.landmark:
        x, y = int(lm.x * w), int(lm.y * h)
        x_min, y_min = min(x_min, x), min(y_min, y)
        x_max, y_max = max(x_max, x), max(y_max, y)
    
    pad = 50
    x_min = max(0, x_min - pad)
    y_min = max(0, y_min - pad)
    x_max = min(w, x_max + pad)
    y_max = min(h, y_max + pad)
    
    if x_max > x_min and y_max > y_min:
        crop = frame[y_min:y_max, x_min:x_max]
        bbox = (x_min, y_min, x_max - x_min, y_max - y_min)
        return crop, bbox, landmarks
    
    return None, None, None

def get_hand_center(landmarks, frame_shape):
    h, w = frame_shape[:2]
    points = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks.landmark]
    center_x = sum(p[0] for p in points) // len(points)
    center_y = sum(p[1] for p in points) // len(points)
    return (center_x, center_y)

def count_fingers(landmarks, frame_shape):
    if not landmarks:
        return 0
    
    h, w = frame_shape[:2]
    points = [(lm.x * w, lm.y * h) for lm in landmarks.landmark]
    
    fingers = 0
    tips = [4, 8, 12, 16, 20]
    pips = [3, 6, 10, 14, 18]
    mcps = [2, 5, 9, 13, 17]
    
    # Thumb
    wrist_x = points[0][0]
    thumb_tip_x = points[tips[0]][0]
    is_right = thumb_tip_x > wrist_x
    
    if is_right:
        if points[tips[0]][0] > points[pips[0]][0]:
            fingers += 1
    else:
        if points[tips[0]][0] < points[pips[0]][0]:
            fingers += 1
    
    # Other fingers
    for i in range(1, 5):
        if points[tips[i]][1] < points[pips[i]][1] - 15 and points[tips[i]][1] < points[mcps[i]][1]:
            fingers += 1
    
    return fingers

def classify_hand(hand_crop):
    """Improved classification with multiple predictions for consistency"""
    if hand_crop is None or hand_crop.size == 0:
        return None, 0.0
    
    if hand_crop.shape[0] < 50 or hand_crop.shape[1] < 50:
        return None, 0.0
    
    try:
        img = cv2.cvtColor(hand_crop, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = img.astype('float32') / 255.0
        
        # Make multiple predictions for stability (Test-Time Augmentation concept)
        predictions = []
        
        # Original
        pred = model.predict(np.expand_dims(img, 0), verbose=0)[0][0]
        predictions.append(pred)
        
        # Slightly augmented versions for robustness
        for _ in range(2):  # 2 additional predictions
            aug_img = img.copy()
            # Small random adjustments
            brightness = np.random.uniform(0.9, 1.1)
            aug_img = np.clip(aug_img * brightness, 0, 1)
            pred = model.predict(np.expand_dims(aug_img, 0), verbose=0)[0][0]
            predictions.append(pred)
        
        # Average predictions for consistency
        avg_pred = np.mean(predictions)
        
        if avg_pred > 0.5:
            return 'palm', avg_pred
        else:
            return 'back', 1 - avg_pred
    except:
        return None, 0.0


# QR CODE SCANNER
def scan_qr_code(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    qr_codes = pyzbar.decode(gray)
    
    if qr_codes:
        qr_data = qr_codes[0].data.decode('utf-8')
        points = qr_codes[0].polygon
        return qr_data, points
    
    return None, None

# UI COMPONENTS
class Button:
    def __init__(self, x, y, w, h, text, color):
        self.x = x
        self.y = y
        self.w = w
        self.h = h
        self.text = text
        self.color = color
        self.hover_start = None
        self.hover_progress = 0.0
    
    def is_hovered(self, point):
        px, py = point
        return (self.x <= px <= self.x + self.w and 
                self.y <= py <= self.y + self.h)
    
    def update_hover(self, is_hovering):
        if is_hovering:
            if self.hover_start is None:
                self.hover_start = time.time()
            else:
                elapsed = time.time() - self.hover_start
                self.hover_progress = min(1.0, elapsed / HOVER_TIME)
        else:
            self.hover_start = None
            self.hover_progress = 0.0
    
    def draw(self, frame):
        overlay = frame.copy()
        cv2.rectangle(overlay, (self.x, self.y), 
                     (self.x + self.w, self.y + self.h), 
                     self.color, -1)
        
        if self.hover_progress > 0:
            progress_h = int(self.h * self.hover_progress)
            cv2.rectangle(overlay, (self.x, self.y + self.h - progress_h),
                         (self.x + self.w, self.y + self.h),
                         (0, 255, 0), -1)
        
        frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
        
        border_color = (0, 255, 0) if self.hover_progress > 0 else (255, 255, 255)
        thickness = 4 if self.hover_progress > 0 else 2
        cv2.rectangle(frame, (self.x, self.y),
                     (self.x + self.w, self.y + self.h),
                     border_color, thickness)
        
        font = cv2.FONT_HERSHEY_SIMPLEX
        text_size = cv2.getTextSize(self.text, font, 1.0, 2)[0]
        text_x = self.x + (self.w - text_size[0]) // 2
        text_y = self.y + (self.h + text_size[1]) // 2
        cv2.putText(frame, self.text, (text_x, text_y),
                   font, 1.0, (255, 255, 255), 2)
        
        return frame

def draw_laser_pointer(frame, point):
    """Red laser pointer"""
    cv2.circle(frame, point, 20, (0, 0, 255), 3)
    cv2.circle(frame, point, 10, (0, 0, 255), -1)
    cv2.line(frame, (point[0] - 30, point[1]), (point[0] + 30, point[1]), (0, 0, 255), 2)
    cv2.line(frame, (point[0], point[1] - 30), (point[0], point[1] + 30), (0, 0, 255), 2)
    return frame

def draw_hand_info(frame, bbox, hand_type, confidence, finger_count, landmarks):
    """Draw professional hand detection info (like previous design)"""
    x, y, w, h = bbox
    
    # Determine color
    color = (0, 255, 0) if hand_type == 'palm' else (0, 165, 255)
    
    # Draw bounding box
    cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)
    
    # Draw hand landmarks (palm lines)
    if landmarks:
        mp_drawing.draw_landmarks(frame, landmarks, mp_hands.HAND_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=color, thickness=2, circle_radius=3),
                                 mp_drawing.DrawingSpec(color=color, thickness=2))
    
    # Info text with background
    info_text = f"{hand_type.upper()} | {finger_count} fingers | {confidence:.0%}"
    text_size = cv2.getTextSize(info_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
    
    # Background for text
    cv2.rectangle(frame, (x, y - 35), (x + text_size[0] + 10, y), (30, 30, 30), -1)
    cv2.putText(frame, info_text, (x + 5, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    
    return frame

def draw_circular_countdown(frame, remaining, total):
    """Circular countdown (like previous design)"""
    overlay = frame.copy()
    h, w = frame.shape[:2]
    
    # Semi-transparent background
    cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
    frame = cv2.addWeighted(frame, 0.4, overlay, 0.6, 0)
    
    center = (w//2, h//2)
    radius = 120
    
    # Background circle
    cv2.circle(frame, center, radius + 15, (50, 50, 50), -1)
    cv2.circle(frame, center, radius + 15, (0, 255, 0), 4)
    
    # Progress arc (circular countdown)
    angle = int(360 * (1 - remaining / total))
    cv2.ellipse(frame, center, (radius, radius), -90, 0, angle, (0, 255, 0), 20)
    
    # Countdown number
    countdown_num = int(remaining) + 1
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size = cv2.getTextSize(str(countdown_num), font, 4, 8)[0]
    text_x = center[0] - text_size[0] // 2
    text_y = center[1] + text_size[1] // 2
    cv2.putText(frame, str(countdown_num), (text_x, text_y), font, 4, (255, 255, 255), 8)
    
    # Capture text
    capture_text = "CAPTURING..."
    text_size = cv2.getTextSize(capture_text, font, 1.2, 3)[0]
    text_x = center[0] - text_size[0] // 2
    cv2.putText(frame, capture_text, (text_x, center[1] + 180), font, 1.2, (0, 255, 0), 3)
    
    return frame

def draw_qr_scanner(frame):
    """QR scanner interface"""
    h, w = frame.shape[:2]
    
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
    frame = cv2.addWeighted(frame, 0.3, overlay, 0.7, 0)
    
    box_size = 400
    box_x = (w - box_size) // 2
    box_y = (h - box_size) // 2
    
    cv2.rectangle(frame, (box_x, box_y), 
                 (box_x + box_size, box_y + box_size),
                 (0, 255, 0), 3)
    
    # Corner markers
    corner_len = 50
    corners = [
        (box_x, box_y),
        (box_x + box_size, box_y),
        (box_x, box_y + box_size),
        (box_x + box_size, box_y + box_size)
    ]
    
    for cx, cy in corners:
        if cx == box_x:
            cv2.line(frame, (cx, cy), (cx + corner_len, cy), (0, 255, 0), 5)
        else:
            cv2.line(frame, (cx, cy), (cx - corner_len, cy), (0, 255, 0), 5)
        
        if cy == box_y:
            cv2.line(frame, (cx, cy), (cx, cy + corner_len), (0, 255, 0), 5)
        else:
            cv2.line(frame, (cx, cy), (cx, cy - corner_len), (0, 255, 0), 5)
    
    # Title
    title = "SCAN YOUR QR CODE"
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size = cv2.getTextSize(title, font, 1.5, 3)[0]
    text_x = (w - text_size[0]) // 2
    cv2.putText(frame, title, (text_x, box_y - 40), font, 1.5, (0, 255, 0), 3)
    
    inst = "Place QR code in the box"
    text_size = cv2.getTextSize(inst, font, 0.8, 2)[0]
    text_x = (w - text_size[0]) // 2
    cv2.putText(frame, inst, (text_x, box_y + box_size + 50), font, 0.8, (200, 200, 200), 2)
    
    return frame


def test_email_configuration():
    """Test email setup before running system"""
    print("\n" + "="*70)
    print("📧 TESTING EMAIL CONFIGURATION")
    print("="*70)
    
    # Validation - check for placeholder values
    if SENDER_EMAIL == "your_email@gmail.com":
        print("❌ ERROR: Email not configured!")
        print("   Please update SENDER_EMAIL in the code")
        return False
    
    if SENDER_PASSWORD == "your_app_password":
        print("❌ ERROR: App password not configured!")
        print("   Please update SENDER_PASSWORD in the code")
        return False
    
    print(f"✓ Sender email: {SENDER_EMAIL}")
    print(f"✓ Password configured: {'*' * 16}")
    
    # Test connection
    try:
        print("\n🔌 Testing Gmail connection...")
        server = smtplib.SMTP('smtp.gmail.com', 587, timeout=10)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.quit()
        print("✅ Email configuration is WORKING!")
        print("="*70 + "\n")
        return True
    except smtplib.SMTPAuthenticationError:
        print("\n❌ LOGIN FAILED!")
        print("Solutions:")
        print("1. Enable 2-Step Verification in Google Account")
        print("2. Generate App Password (not regular password)")
        print("3. Remove spaces from app password")
        print("="*70 + "\n")
        return False
    except Exception as e:
        print(f"\n❌ Connection error: {e}")
        print("="*70 + "\n")
        return False
    
    
# MAIN SYSTEM
def run_system():
    # Load students from CSV
    students_db = load_students_from_csv()
    
    if len(students_db) == 0:
        print("❌ No students loaded! Check CSV file path.")
        return
    
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    class State:
        WAITING = "waiting"
        STABLE = "stable"
        COUNTDOWN = "countdown"
        SHOW_PHOTO = "show_photo"
        QR_SCAN = "qr_scan"
        CONFIRM_IDENTITY = "confirm_identity"
    
    state = State.WAITING
    stable_start = 0
    countdown_start = 0
    captured_image = None
    picture_id = None
    scanned_student = None
    
    # Buttons
    download_btn = None
    cancel_btn = None
    yes_btn = None
    no_btn = None
    rescan_btn = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        display = frame.copy()
        
        hand_crop, bbox, landmarks = detect_hand(frame)
        
        # ===== WAITING =====
        if state == State.WAITING:
            overlay = display.copy()
            h, w = display.shape[:2]
            cv2.rectangle(overlay, (0, 0), (w, 100), (50, 50, 50), -1)
            display = cv2.addWeighted(display, 0.7, overlay, 0.3, 0)
            
            text = "Show PALM with 5 fingers to start"
            font = cv2.FONT_HERSHEY_SIMPLEX
            text_size = cv2.getTextSize(text, font, 1.2, 2)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, text, (text_x, 60), font, 1.2, (0, 255, 255), 2)
            
            if hand_crop is not None and bbox is not None:
                hand_type, conf = classify_hand(hand_crop)
                fingers = count_fingers(landmarks, frame.shape)
                
                # Draw hand info (like previous design)
                display = draw_hand_info(display, bbox, hand_type, conf, fingers, landmarks)
                
                if hand_type == 'palm' and fingers == 5 and conf >= CONFIDENCE_THRESHOLD:
                    state = State.STABLE
                    stable_start = time.time()
        
        # ===== STABLE =====
        elif state == State.STABLE:
            elapsed = time.time() - stable_start
            remaining = STABLE_TIME - elapsed
            
            if remaining > 0:
                h, w = display.shape[:2]
                progress = elapsed / STABLE_TIME
                
                # Progress bar
                bar_w = int(w * 0.6)
                bar_h = 30
                bar_x = (w - bar_w) // 2
                bar_y = 50
                
                cv2.rectangle(display, (bar_x, bar_y), 
                             (bar_x + bar_w, bar_y + bar_h), (100, 100, 100), -1)
                progress_w = int(bar_w * progress)
                cv2.rectangle(display, (bar_x, bar_y),
                             (bar_x + progress_w, bar_y + bar_h), (255, 165, 0), -1)
                
                text = f"Hold steady... {remaining:.1f}s"
                font = cv2.FONT_HERSHEY_SIMPLEX
                text_size = cv2.getTextSize(text, font, 1.0, 2)[0]
                text_x = (w - text_size[0]) // 2
                cv2.putText(display, text, (text_x, bar_y - 10), font, 1.0, (255, 165, 0), 2)
                
                # Continue showing hand info
                if hand_crop is not None and bbox is not None:
                    hand_type, conf = classify_hand(hand_crop)
                    fingers = count_fingers(landmarks, frame.shape)
                    display = draw_hand_info(display, bbox, hand_type, conf, fingers, landmarks)
                    
                    if not (hand_type == 'palm' and fingers == 5 and conf >= CONFIDENCE_THRESHOLD):
                        state = State.WAITING
                else:
                    state = State.WAITING

            else:
                state = State.COUNTDOWN
                countdown_start = time.time()
        
        # ===== COUNTDOWN =====
        elif state == State.COUNTDOWN:
            elapsed = time.time() - countdown_start
            remaining = COUNTDOWN_TIME - elapsed
            
            if remaining > 0:
                display = draw_circular_countdown(display, remaining, COUNTDOWN_TIME)
            else:
                # CAPTURE!
                captured_image = frame.copy()
                picture_id = datetime.now().strftime("%Y%m%d_%H%M%S")

                state = State.SHOW_PHOTO
                
                # Create buttons
                h, w = display.shape[:2]
                btn_w = 300
                btn_h = 100
                spacing = 80
                
                download_btn = Button(w//2 - btn_w - spacing//2, h - 150, 
                                     btn_w, btn_h, "DOWNLOAD", (0, 150, 0))
                cancel_btn = Button(w//2 + spacing//2, h - 150,
                                   btn_w, btn_h, "CANCEL", (150, 0, 0))
        
        # ===== SHOW PHOTO =====
        elif state == State.SHOW_PHOTO:
            h, w = display.shape[:2]
            
            # Dark background
            overlay = display.copy()
            cv2.rectangle(overlay, (0, 0), (w, h), (20, 20, 20), -1)
            display = cv2.addWeighted(display, 0.2, overlay, 0.8, 0)
            
            # Show captured image
            img_w = 600
            img_h = 450
            img_x = (w - img_w) // 2
            img_y = 80
            
            cap_resized = cv2.resize(captured_image, (img_w, img_h))
            display[img_y:img_y+img_h, img_x:img_x+img_w] = cap_resized
            cv2.rectangle(display, (img_x, img_y),
                         (img_x + img_w, img_y + img_h), (0, 255, 0), 3)
            
            # Title
            font = cv2.FONT_HERSHEY_SIMPLEX
            title = "PHOTO CAPTURED!"
            text_size = cv2.getTextSize(title, font, 1.5, 3)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, title, (text_x, img_y - 30), font, 1.5, (0, 255, 0), 3)
            
            # Draw buttons
            display = download_btn.draw(display)
            display = cancel_btn.draw(display)
            
            # Instructions
            inst = "Hover over button with hand for 2 seconds"
            text_size = cv2.getTextSize(inst, font, 0.8, 2)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, inst, (text_x, h - 50), font, 0.8, (150, 150, 150), 2)
            
            # Hand tracking
            if landmarks:
                hand_center = get_hand_center(landmarks, frame.shape)
                display = draw_laser_pointer(display, hand_center)
                
                download_btn.update_hover(download_btn.is_hovered(hand_center))
                cancel_btn.update_hover(cancel_btn.is_hovered(hand_center))
                
                if download_btn.hover_progress >= 1.0:
                    state = State.QR_SCAN
                    time.sleep(0.5)
                
                elif cancel_btn.hover_progress >= 1.0:
                    state = State.WAITING
                    captured_image = None
                    picture_id = None
                    time.sleep(0.5)
        
        # ===== QR SCAN =====
        elif state == State.QR_SCAN:
            display = draw_qr_scanner(display)
            
            # Add "Back" button to capture new photo
            h, w = display.shape[:2]
            if not hasattr(run_system, 'back_btn_qr'):
                btn_w = 250
                btn_h = 80
                run_system.back_btn_qr = Button(w - btn_w - 50, 50, btn_w, btn_h, "NEW PHOTO", (100, 100, 200))
            
            display = run_system.back_btn_qr.draw(display)
            
            # Instructions for back button
            font = cv2.FONT_HERSHEY_SIMPLEX
            inst = "Hover on 'NEW PHOTO' to capture again"
            text_size = cv2.getTextSize(inst, font, 0.7, 2)[0]
            cv2.putText(display, inst, (w - text_size[0] - 50, 150), font, 0.7, (200, 200, 200), 2)
            
            # Check back button with hand
            if landmarks:
                hand_center = get_hand_center(landmarks, frame.shape)
                display = draw_laser_pointer(display, hand_center)
                
                run_system.back_btn_qr.update_hover(run_system.back_btn_qr.is_hovered(hand_center))
                
                if run_system.back_btn_qr.hover_progress >= 1.0:
                    state = State.WAITING
                    captured_image = None
                    picture_id = None
                    scanned_student = None
                    time.sleep(0.5)
                    continue
            
            qr_data, qr_points = scan_qr_code(frame)
            
            if qr_data:
                student = get_student_by_qr(qr_data, students_db) 
                
                if student:
                    scanned_student = student
                    
                    # Create confirmation buttons - BETTER POSITIONING
                    btn_w = 280  # Slightly wider
                    btn_h = 90   # Slightly shorter
                    spacing = 100  # More spacing
                    
                    yes_btn = Button(w//2 - btn_w - spacing//2, h - 180,  # Higher position
                                    btn_w, btn_h, "YES", (0, 200, 0))
                    no_btn = Button(w//2 + spacing//2, h - 180,  # Higher position
                                btn_w, btn_h, "NO", (200, 0, 0))
                    state = State.CONFIRM_IDENTITY
                    time.sleep(1)
                else:
                    text = "QR CODE NOT RECOGNIZED!"
                    text_size = cv2.getTextSize(text, font, 1.5, 3)[0]
                    text_x = (w - text_size[0]) // 2
                    cv2.putText(display, text, (text_x, h - 250), font, 1.5, (0, 0, 255), 3)
        
        # ===== CONFIRM IDENTITY =====
        elif state == State.CONFIRM_IDENTITY:
            h, w = display.shape[:2]
            
            # Dark background
            overlay = display.copy()
            cv2.rectangle(overlay, (0, 0), (w, h), (20, 20, 20), -1)
            display = cv2.addWeighted(display, 0.2, overlay, 0.8, 0)           
        
            # Show captured image (adjusted size and position)
            img_w = 400  # Smaller width
            img_h = 300  # Smaller height
            img_x = (w - img_w) // 2
            img_y = 50  # Higher position
            
            cap_resized = cv2.resize(captured_image, (img_w, img_h))
            display[img_y:img_y+img_h, img_x:img_x+img_w] = cap_resized
            cv2.rectangle(display, (img_x, img_y),
                        (img_x + img_w, img_y + img_h), (0, 255, 0), 3)
            
            # Info section with better spacing
            font = cv2.FONT_HERSHEY_SIMPLEX
            info_start_y = img_y + img_h + 40  # Space after image
            
            # Title
            title_text = "Confirm Your Identity"
            text_size = cv2.getTextSize(title_text, font, 1.2, 3)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, title_text, (text_x, info_start_y), 
                    font, 1.2, (0, 255, 255), 3)
            
            # Student name with background box
            name_text = f"Name: {scanned_student['name']}"
            text_size = cv2.getTextSize(name_text, font, 1.0, 2)[0]
            
            # Background for name
            box_padding = 15
            name_box_x = (w - text_size[0]) // 2 - box_padding
            name_box_y = info_start_y + 40
            cv2.rectangle(display, 
                        (name_box_x, name_box_y - text_size[1] - 5),
                        (name_box_x + text_size[0] + box_padding * 2, name_box_y + 10),
                        (50, 50, 50), -1)
            
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, name_text, (text_x, name_box_y), 
                    font, 1.0, (255, 255, 255), 2)
            
            # Email with background box
            email_text = f"Email: {scanned_student['email']}"
            text_size = cv2.getTextSize(email_text, font, 0.8, 2)[0]
            
            email_box_x = (w - text_size[0]) // 2 - box_padding
            email_box_y = name_box_y + 50
            cv2.rectangle(display, 
                        (email_box_x, email_box_y - text_size[1] - 5),
                        (email_box_x + text_size[0] + box_padding * 2, email_box_y + 10),
                        (50, 50, 50), -1)
            
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, email_text, (text_x, email_box_y), 
                    font, 0.8, (200, 200, 200), 2)
            
            # Student ID
            id_text = f"ID: {scanned_student['student_id']}"
            text_size = cv2.getTextSize(id_text, font, 0.8, 2)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, id_text, (text_x, email_box_y + 40), 
                    font, 0.8, (150, 150, 150), 2)
            
            # Draw buttons (positioned higher to avoid overlap)
            display = yes_btn.draw(display)
            display = no_btn.draw(display)
            
            # Instructions (above buttons)
            inst = "Move your hand to YES or NO (hold 2s)"
            text_size = cv2.getTextSize(inst, font, 0.7, 2)[0]
            text_x = (w - text_size[0]) // 2
            cv2.putText(display, inst, (text_x, h - 270), font, 0.7, (255, 255, 0), 2)
            
            # Hand tracking
            if landmarks:
                hand_center = get_hand_center(landmarks, frame.shape)
                display = draw_laser_pointer(display, hand_center)
                
                yes_btn.update_hover(yes_btn.is_hovered(hand_center))
                no_btn.update_hover(no_btn.is_hovered(hand_center))
                
                if yes_btn.hover_progress >= 1.0:

                    filename = f"graduation_{picture_id}.jpg"
                    filepath = os.path.join(CAPTURES_FOLDER, filename)
                    cv2.imwrite(filepath, captured_image)
                    
                    # Show sending message
                    overlay = display.copy()
                    cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
                    display = cv2.addWeighted(display, 0.3, overlay, 0.7, 0)
                    
                    sending_text = "SENDING EMAIL..."
                    text_size = cv2.getTextSize(sending_text, font, 2.0, 4)[0]
                    text_x = (w - text_size[0]) // 2
                    text_y = h // 2
                    cv2.putText(display, sending_text, (text_x, text_y), 
                               font, 2.0, (0, 255, 255), 4)
                    cv2.imshow('Graduation Photo System', display)
                    cv2.waitKey(1)
                    
                    success = send_graduation_email(
                        scanned_student['email'],
                        scanned_student['name'],
                        picture_id,
                        filepath
                    )
                    
                    if success:
                        print(f"\n{'='*70}")
                        print(f"✅ EMAIL SENT SUCCESSFULLY!")
                        print(f"{'='*70}")
                        print(f"Student: {scanned_student['name']}")
                        print(f"Email: {scanned_student['email']}")
                        print(f"Picture ID: {picture_id}")
                        print(f"{'='*70}\n")
                        
                        # IMPROVED SUCCESS SCREEN
                        success_frame = frame.copy()
                        h, w = success_frame.shape[:2]
                        
                        # Dark green background overlay
                        overlay = success_frame.copy()
                        cv2.rectangle(overlay, (0, 0), (w, h), (0, 80, 0), -1)
                        success_frame = cv2.addWeighted(success_frame, 0.2, overlay, 0.8, 0)
                        
                        # Success box
                        box_w = 900
                        box_h = 500
                        box_x = (w - box_w) // 2
                        box_y = (h - box_h) // 2
                        
                        # White box background
                        cv2.rectangle(success_frame, (box_x, box_y), 
                                    (box_x + box_w, box_y + box_h), 
                                    (255, 255, 255), -1)
                        cv2.rectangle(success_frame, (box_x, box_y), 
                                    (box_x + box_w, box_y + box_h), 
                                    (0, 255, 0), 8)
                        
                        # Large checkmark circle
                        circle_center = (w // 2, box_y + 120)
                        cv2.circle(success_frame, circle_center, 80, (0, 200, 0), -1)
                        cv2.circle(success_frame, circle_center, 80, (0, 255, 0), 5)
                        
                        # Checkmark symbol
                        check_pts = np.array([
                            [circle_center[0] - 35, circle_center[1]],
                            [circle_center[0] - 10, circle_center[1] + 30],
                            [circle_center[0] + 40, circle_center[1] - 30]
                        ], np.int32)
                        cv2.polylines(success_frame, [check_pts], False, (255, 255, 255), 12)
                        
                        # Success text
                        success_text = "EMAIL SENT!"
                        text_size = cv2.getTextSize(success_text, font, 2.0, 4)[0]
                        text_x = (w - text_size[0]) // 2
                        text_y = box_y + 250
                        cv2.putText(success_frame, success_text, (text_x, text_y), 
                                font, 2.0, (0, 150, 0), 4)
                        
                        # Student info
                        info_y = text_y + 60
                        line_height = 45
                        
                        info_lines = [
                            f"Name: {scanned_student['name']}",
                            f"Email: {scanned_student['email']}",
                            f"Picture ID: {picture_id}"
                        ]
                        
                        for i, line in enumerate(info_lines):
                            text_size = cv2.getTextSize(line, font, 0.9, 2)[0]
                            text_x = (w - text_size[0]) // 2
                            cv2.putText(success_frame, line, (text_x, info_y + i * line_height), 
                                    font, 0.9, (50, 50, 50), 2)
                        
                        # Bottom message
                        bottom_msg = "Thank you! Starting next capture..."
                        text_size = cv2.getTextSize(bottom_msg, font, 0.8, 2)[0]
                        text_x = (w - text_size[0]) // 2
                        cv2.putText(success_frame, bottom_msg, 
                                (text_x, box_y + box_h - 30), 
                                font, 0.8, (100, 100, 100), 2)
                        
                        cv2.imshow('Graduation Photo System', success_frame)
                        cv2.waitKey(3000)
                    
                    else:
                        # IMPROVED ERROR SCREEN
                        error_frame = frame.copy()
                        h, w = error_frame.shape[:2]
                        
                        # Dark red background
                        overlay = error_frame.copy()
                        cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 80), -1)
                        error_frame = cv2.addWeighted(error_frame, 0.2, overlay, 0.8, 0)
                        
                        # Error box
                        box_w = 800
                        box_h = 400
                        box_x = (w - box_w) // 2
                        box_y = (h - box_h) // 2
                        
                        # White box
                        cv2.rectangle(error_frame, (box_x, box_y), 
                                    (box_x + box_w, box_y + box_h), 
                                    (255, 255, 255), -1)
                        cv2.rectangle(error_frame, (box_x, box_y), 
                                    (box_x + box_w, box_y + box_h), 
                                    (0, 0, 255), 8)
                        
                        # Error symbol (X)
                        circle_center = (w // 2, box_y + 100)
                        cv2.circle(error_frame, circle_center, 70, (200, 0, 0), -1)
                        cv2.circle(error_frame, circle_center, 70, (0, 0, 255), 5)
                        
                        # X mark
                        cv2.line(error_frame, 
                                (circle_center[0] - 35, circle_center[1] - 35),
                                (circle_center[0] + 35, circle_center[1] + 35),
                                (255, 255, 255), 12)
                        cv2.line(error_frame, 
                                (circle_center[0] + 35, circle_center[1] - 35),
                                (circle_center[0] - 35, circle_center[1] + 35),
                                (255, 255, 255), 12)
                        
                        # Error text
                        error_text = "EMAIL FAILED!"
                        text_size = cv2.getTextSize(error_text, font, 2.0, 4)[0]
                        text_x = (w - text_size[0]) // 2
                        text_y = box_y + 220
                        cv2.putText(error_frame, error_text, (text_x, text_y), 
                                font, 2.0, (150, 0, 0), 4)
                        
                        # Help message
                        help_msg = "Please contact support"
                        text_size = cv2.getTextSize(help_msg, font, 0.9, 2)[0]
                        text_x = (w - text_size[0]) // 2
                        cv2.putText(error_frame, help_msg, (text_x, text_y + 60), 
                                font, 0.9, (100, 100, 100), 2)
                        
                        cv2.imshow('Graduation Photo System', error_frame)
                        cv2.waitKey(3000)
                    
                    # Reset
                    state = State.WAITING
                    captured_image = None
                    picture_id = None
                    scanned_student = None
                    time.sleep(1)
                
                elif no_btn.hover_progress >= 1.0:
                    # Wrong identity - go back to QR scan
                    state = State.QR_SCAN
                    scanned_student = None
                    time.sleep(0.5)
        
        # Display
        cv2.imshow('Graduation Photo System', display)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Cleanup
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    print("\n✅ System closed")

# RUN
if __name__ == "__main__":

    # Test email before starting
    print("\nTesting email configuration...")
    if not test_email_configuration():
        print("The system will still run, but emails won't send.")
        response = input("\nContinue anyway? (y/n): ")
        if response.lower() != 'y':
            print("Please configure email first. Exiting...")
            exit()
    
    run_system()

🔄 Loading model...
✓ Model loaded

Testing email configuration...

📧 TESTING EMAIL CONFIGURATION
✓ Sender email: xinyaoteoh@gmail.com
✓ Password configured: ****************

🔌 Testing Gmail connection...
✅ Email configuration is WORKING!

✓ Loaded 4 students from CSV

✅ System closed
